# Digital Twin for Diabetic Retinopathy
## Graph-Aware Biomarker Detection, Progression Modeling & Virtual Drug Testing

This notebook implements a three-layer digital twin framework for diabetic retinopathy:

1. **Layer 1 — Biomarker Detection**: GNN-based multi-label biomarker detection from OCT scans (CNN, GCN, Signed GCN, Prior-GAT)
2. **Layer 2 — Temporal Progression**: Graph-constrained LSTM for disease trajectory prediction
3. **Layer 3 — Virtual Drug Testing**: Treatment simulation engine with counterfactual comparison

**Dataset**: OLIVES (PRIME + TREX DME clinical trials, 9,396 OCT slices, 87 patients)

---

## Setup & Imports

In [ ]:
import os
import sys
import json
import math
import copy
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

warnings.filterwarnings('ignore', category=FutureWarning)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Configuration

In [ ]:
# === Dataset paths ===
KAGGLE_LABELS_DIR = '/kaggle/input/datasets/saiamarnathg/datalabels'
LABEL_CSV = os.path.join(KAGGLE_LABELS_DIR, 'Biomarker_Clinical_Data_Images.csv')
OLIVES_ROOT = '/kaggle/input/datasets/abhinavk1609/olives/OLIVES'
PRIME_ROOT = os.path.join(OLIVES_ROOT, 'Prime_FULL', 'Prime_FULL')
TREX_ROOT = os.path.join(OLIVES_ROOT, 'TREX_DME', 'TREX DME')

# === Biomarkers ===
BIOMARKERS = ['IRHRF', 'PAVF', 'FAVF', 'IRF', 'DRT/ME', 'VD']
N_BM = len(BIOMARKERS)

# === Quick debug mode ===
QUICK_DEBUG = False

# === Hyperparameters ===
IMG_SIZE = 224
NODE_DIM = 128
N_GNN_LAYERS = 2
GNN_DROPOUT = 0.3
GRAPH_TOP_K = 3
LR = 1e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 16 if not QUICK_DEBUG else 8
MAX_EPOCHS = 40 if not QUICK_DEBUG else 2
EARLY_STOP_PATIENCE = 6
GRAD_CLIP = 2.0
SEEDS = [42, 123, 2024] if not QUICK_DEBUG else [42]
SPLIT_SEED = 2025

# === Progression model ===
PROG_HIDDEN = 64
PROG_LAYERS = 2
PROG_EPOCHS = 100 if not QUICK_DEBUG else 5
PROG_LR = 5e-4

# === Treatment types ===
TREATMENTS = ['anti_vegf', 'steroid', 'laser', 'observation']

# === Output ===
OUT_DIR = './outputs'
CKPT_DIR = os.path.join(OUT_DIR, 'checkpoints')
METRICS_DIR = os.path.join(OUT_DIR, 'metrics')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

print('Config ready. QUICK_DEBUG:', QUICK_DEBUG)

---
# LAYER 1: Biomarker Detection Engine

Ported from the core6_biomarker_graph notebook. Detects six retinal biomarkers from OCT scans using a ResNet-18 backbone with GNN-based co-occurrence modeling.

### 1.1 Data Loading & Preprocessing

In [ ]:
import re

def normalize_col(c):
    return re.sub(r'[^a-z0-9]', '', str(c).lower())

def find_column(df, candidates):
    norm_map = {normalize_col(c): c for c in df.columns}
    for cand in candidates:
        key = normalize_col(cand)
        if key in norm_map:
            return norm_map[key]
    for cand in candidates:
        key = normalize_col(cand)
        for k, orig in norm_map.items():
            if key in k or k in key:
                return orig
    return None

BM_COL_CANDIDATES = {
    'IRHRF': ['IR HRF', 'IRHRF'], 'PAVF': ['Partially attached vitreous face', 'PAVF'],
    'FAVF': ['Fully attached vitreous face', 'FAVF'], 'IRF': ['Fluid (IRF)', 'IRF'],
    'DRT/ME': ['DRT/ME', 'DRT ME', 'DRT'], 'VD': ['Vitreous debris', 'VD'],
}

df_raw = pd.read_csv(LABEL_CSV)
print(f'Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns')

PATH_COL = find_column(df_raw, ['Path', 'Image Path'])
PATIENT_COL = find_column(df_raw, ['Patient_ID', 'PatientID'])
EYE_COL = find_column(df_raw, ['Eye_ID', 'EyeID'])
VISIT_COL = find_column(df_raw, ['Week', 'Visit', 'Timepoint'])

resolved_bm = {bm: find_column(df_raw, cands) for bm, cands in BM_COL_CANDIDATES.items()}
biomarker_cols = [resolved_bm[b] for b in BIOMARKERS]
print('Biomarker columns:', {b: resolved_bm[b] for b in BIOMARKERS})

# Drop missing labels
df = df_raw.dropna(subset=biomarker_cols).reset_index(drop=True)
for c in biomarker_cols:
    df[c] = df[c].astype(int)
print(f'After dropping missing: {len(df)} rows')

In [ ]:
# Image path resolution
def _normalize_prefix(s):
    return re.sub(r'[^A-Za-z]', '', s).upper()

def resolve_image_path(raw_path):
    p = str(raw_path).strip().replace('\\', '/')
    if p.startswith('/'): p = p[1:]
    parts = [x for x in p.split('/') if x]
    if not parts: return None
    prefix = _normalize_prefix(parts[0])
    rest = parts[1:]
    roots = [TREX_ROOT, PRIME_ROOT] if prefix.startswith('TREX') else [PRIME_ROOT, TREX_ROOT]
    for root in roots:
        cand = os.path.join(root, *rest) if rest else root
        if os.path.exists(cand): return cand
    return os.path.join(roots[0], *rest)

df['resolved_path'] = df[PATH_COL].apply(resolve_image_path)
exists = df['resolved_path'].apply(lambda p: p is not None and os.path.exists(str(p)))
print(f'Resolved {exists.sum()}/{len(df)} image paths ({100*exists.mean():.1f}%)')
df = df[exists].reset_index(drop=True)

### 1.2 Patient-Level Split

In [ ]:
def make_patient_split(df, patient_col, bm_cols, train_f=0.7, val_f=0.15, seed=0, n_trials=200):
    rng = np.random.RandomState(seed)
    patients = df[patient_col].unique().tolist()
    counts = df.groupby(patient_col).size()
    overall = df[bm_cols].mean().values
    target_train = train_f * counts.sum()
    target_val = val_f * counts.sum()
    best, best_score = None, np.inf
    for _ in range(n_trials):
        perm = patients.copy(); rng.shuffle(perm)
        tr, va, te = [], [], []
        tr_n = va_n = 0
        for pid in perm:
            c = counts[pid]
            if tr_n < target_train: tr.append(pid); tr_n += c
            elif va_n < target_val: va.append(pid); va_n += c
            else: te.append(pid)
        masks = [df[patient_col].isin(ids) for ids in [tr, va, te]]
        if any(m.sum() == 0 for m in masks): continue
        score = sum(np.sum((df.loc[m, bm_cols].mean().values - overall)**2) for m in masks)
        if score < best_score: best_score = score; best = (tr, va, te)
    tr, va, te = best
    return (df[df[patient_col].isin(tr)].reset_index(drop=True),
            df[df[patient_col].isin(va)].reset_index(drop=True),
            df[df[patient_col].isin(te)].reset_index(drop=True))

df_train, df_val, df_test = make_patient_split(df, PATIENT_COL, biomarker_cols, seed=SPLIT_SEED)
print(f'Split: train={len(df_train)}, val={len(df_val)}, test={len(df_test)}')
print(f'Patients: train={df_train[PATIENT_COL].nunique()}, val={df_val[PATIENT_COL].nunique()}, test={df_test[PATIENT_COL].nunique()}')

# Verify no leakage
assert set(df_train[PATIENT_COL]) & set(df_test[PATIENT_COL]) == set()
assert set(df_train[PATIENT_COL]) & set(df_val[PATIENT_COL]) == set()
print('No patient leakage.')

### 1.3 NPMI Co-occurrence Graph

In [ ]:
def compute_npmi(df_train, bm_cols, eps=1e-12):
    X = df_train[bm_cols].values.astype(np.float64)
    p = X.mean(axis=0)
    k = X.shape[1]
    npmi = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            if i == j: npmi[i,j] = 1.0; continue
            p_ij = np.mean((X[:,i]==1) & (X[:,j]==1))
            if p[i] < eps or p[j] < eps: continue
            if p_ij < eps: npmi[i,j] = -1.0; continue
            pmi = math.log(p_ij / (p[i]*p[j]))
            npmi[i,j] = np.clip(pmi / (-math.log(p_ij)), -1, 1)
    return npmi, p

npmi, p_train = compute_npmi(df_train, biomarker_cols)
npmi_df = pd.DataFrame(npmi, index=BIOMARKERS, columns=BIOMARKERS)
print('NPMI matrix (train):')
print(npmi_df.round(3))

# Visualize
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(npmi_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Biomarker Co-occurrence (NPMI)')
plt.tight_layout()
plt.savefig(os.path.join(METRICS_DIR, 'npmi_heatmap.png'), dpi=150)
plt.show()

In [ ]:
# Graph sparsification & adjacency matrices
def sparsify_graph(npmi, top_k=3):
    k = npmi.shape[0]
    mask = np.zeros((k,k), dtype=bool)
    for i in range(k):
        row = npmi[i].copy(); row[i] = -np.inf
        mask[i, np.argsort(-np.abs(row))[:top_k]] = True
    mask = mask | mask.T; np.fill_diagonal(mask, False)
    signed = np.where(mask, npmi, 0.0)
    pos = np.where(signed > 0, signed, 0.0)
    neg = np.where(signed < 0, signed, 0.0)
    return signed, pos, neg, mask

def norm_adj(A, self_loop=True):
    k = A.shape[0]
    A_hat = A + np.eye(k) if self_loop else A.copy()
    deg = np.sum(np.abs(A_hat), axis=1); deg[deg==0] = 1.0
    D = np.diag(1.0/np.sqrt(deg))
    return (D @ A_hat @ D).astype(np.float32)

signed, pos, neg, sym_mask = sparsify_graph(npmi, GRAPH_TOP_K)
graph_bundle = {
    'npmi': npmi, 'signed_graph': signed.astype(np.float32),
    'pos_adj_norm': norm_adj(pos), 'pos_adj_norm_nosl': norm_adj(pos, False),
    'neg_adj_norm_nosl': norm_adj(neg, False),
    'gat_edge_mask': sym_mask | np.eye(N_BM, dtype=bool),
}
print(f'Graph: {int(sym_mask.sum()//2)} undirected edges')

### 1.4 Models (CNN, GCN, Signed GCN, Prior-GAT)

In [ ]:
# All model code from layer1_biomarker/models.py — included inline for Kaggle

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(4),
    transforms.RandomAffine(0, translate=(0.02, 0.02), scale=(0.99, 1.01)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class OCTDataset(Dataset):
    def __init__(self, df, path_col, bm_cols, transform=None):
        self.df = df.reset_index(drop=True)
        self.path_col, self.bm_cols, self.transform = path_col, bm_cols, transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try: img = Image.open(resolve_image_path(row[self.path_col])).convert('L').convert('RGB')
        except: img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        if self.transform: img = self.transform(img)
        labels = row[self.bm_cols].values.astype(np.float32)
        return img, torch.from_numpy(labels)

def build_resnet18():
    try: net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    except: net = models.resnet18(weights=None)
    return nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool, net.layer1, net.layer2, net.layer3, net.layer4), 512

class SpatialBackbone(nn.Module):
    def __init__(self): super().__init__(); self.body, self.out_dim = build_resnet18()
    def forward(self, x): return self.body(x)

class BiomarkerEncoder(nn.Module):
    def __init__(self, in_ch=512, node_dim=128, n_bm=N_BM):
        super().__init__()
        self.n = n_bm
        self.query = nn.Parameter(torch.randn(n_bm, in_ch) * 0.02)
        self.value_proj = nn.Linear(in_ch, node_dim)
        self.node_emb = nn.Embedding(n_bm, node_dim)
    def forward(self, fmap):
        B, C, H, W = fmap.shape
        tokens = fmap.flatten(2).transpose(1,2)
        scores = torch.einsum('bsc,nc->bns', tokens, self.query) / math.sqrt(C)
        attn = torch.softmax(scores, dim=-1)
        pooled = torch.einsum('bns,bsc->bnc', attn, tokens)
        ids = torch.arange(self.n, device=fmap.device)
        return self.value_proj(pooled) + self.node_emb(ids).unsqueeze(0), attn

class GCNLayer(nn.Module):
    def __init__(self, dim, adj):
        super().__init__()
        self.register_buffer('adj', torch.as_tensor(adj, dtype=torch.float32))
        self.lin = nn.Linear(dim, dim); self.norm = nn.LayerNorm(dim)
    def forward(self, h): return F.relu(self.norm(self.lin(torch.einsum('ij,bjd->bid', self.adj, h))))

class SignedGCNLayer(nn.Module):
    def __init__(self, dim, pos_adj, neg_adj):
        super().__init__()
        self.register_buffer('pos', torch.as_tensor(pos_adj, dtype=torch.float32))
        self.register_buffer('neg', torch.as_tensor(neg_adj, dtype=torch.float32))
        self.self_lin = nn.Linear(dim, dim); self.pos_lin = nn.Linear(dim, dim)
        self.neg_lin = nn.Linear(dim, dim); self.norm = nn.LayerNorm(dim)
    def forward(self, h):
        return F.relu(self.norm(self.self_lin(h) + self.pos_lin(torch.einsum('ij,bjd->bid', self.pos, h)) + self.neg_lin(torch.einsum('ij,bjd->bid', self.neg, h))))

class PriorGATLayer(nn.Module):
    def __init__(self, dim, npmi_prior, edge_mask):
        super().__init__()
        self.register_buffer('npmi', torch.as_tensor(npmi_prior, dtype=torch.float32))
        self.register_buffer('mask', torch.as_tensor(edge_mask, dtype=torch.bool))
        self.q = nn.Linear(dim, dim); self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim); self.s = nn.Linear(dim, dim)
        self.alpha = nn.Parameter(torch.tensor(0.5)); self.norm = nn.LayerNorm(dim)
    def forward(self, h, return_attn=False):
        B, N, D = h.shape
        Q, K, V = self.q(h), self.k(h), self.v(h)
        scores = torch.einsum('bid,bjd->bij', Q, K) / math.sqrt(D) + self.alpha * self.npmi.unsqueeze(0)
        scores = scores.masked_fill(~self.mask.unsqueeze(0).expand(B,-1,-1), float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        out = F.relu(self.norm(self.s(h) + torch.einsum('bij,bjd->bid', attn, V)))
        return (out, attn) if return_attn else out

class CNNBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = SpatialBackbone(); self.pool = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, N_BM))
    def forward(self, x): return self.mlp(self.pool(self.backbone(x)).flatten(1)), {}

class GCNModel(nn.Module):
    def __init__(self, adj):
        super().__init__()
        self.backbone = SpatialBackbone(); self.enc = BiomarkerEncoder(512, NODE_DIM)
        self.layers = nn.ModuleList([GCNLayer(NODE_DIM, adj) for _ in range(N_GNN_LAYERS)])
        self.drop = nn.Dropout(GNN_DROPOUT); self.cls = nn.Linear(NODE_DIM, 1)
    def forward(self, x):
        h, sa = self.enc(self.backbone(x))
        for l in self.layers: h = self.drop(l(h))
        return self.cls(h).squeeze(-1), {'spatial_attn': sa}

class SignedGCNModel(nn.Module):
    def __init__(self, pos_adj, neg_adj):
        super().__init__()
        self.backbone = SpatialBackbone(); self.enc = BiomarkerEncoder(512, NODE_DIM)
        self.layers = nn.ModuleList([SignedGCNLayer(NODE_DIM, pos_adj, neg_adj) for _ in range(N_GNN_LAYERS)])
        self.drop = nn.Dropout(GNN_DROPOUT); self.cls = nn.Linear(NODE_DIM, 1)
    def forward(self, x):
        h, sa = self.enc(self.backbone(x))
        for l in self.layers: h = self.drop(l(h))
        return self.cls(h).squeeze(-1), {'spatial_attn': sa}

class PriorGATModel(nn.Module):
    def __init__(self, npmi_prior, edge_mask):
        super().__init__()
        self.backbone = SpatialBackbone(); self.enc = BiomarkerEncoder(512, NODE_DIM)
        self.layers = nn.ModuleList([PriorGATLayer(NODE_DIM, npmi_prior, edge_mask) for _ in range(N_GNN_LAYERS)])
        self.drop = nn.Dropout(GNN_DROPOUT); self.cls = nn.Linear(NODE_DIM, 1)
    def forward(self, x, return_attn=False):
        h, sa = self.enc(self.backbone(x))
        ga = None
        for i, l in enumerate(self.layers):
            if return_attn and i == len(self.layers)-1: h, ga = l(h, True)
            else: h = l(h)
            h = self.drop(h)
        extras = {'spatial_attn': sa}
        if ga is not None: extras['graph_attn'] = ga
        return self.cls(h).squeeze(-1), extras

def build_model(name, gb):
    if name == 'cnn': return CNNBaseline()
    if name == 'gcn': return GCNModel(gb['pos_adj_norm'])
    if name == 'signed_gcn': return SignedGCNModel(gb['pos_adj_norm_nosl'], gb['neg_adj_norm_nosl'])
    if name == 'prior_gat': return PriorGATModel(gb['signed_graph'], gb['gat_edge_mask'])
    raise ValueError(name)

print('All Layer 1 models defined.')

### 1.5 Training & Evaluation

In [ ]:
def sigmoid(x): return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

def compute_metrics(y_true, logits, thresholds=None):
    probs = sigmoid(logits)
    preds = (probs >= (thresholds[None,:] if thresholds is not None else 0.5)).astype(int)
    per_f1 = [f1_score(y_true[:,i], preds[:,i], zero_division=0) for i in range(y_true.shape[1])]
    return {'macro_f1': np.mean(per_f1), 'micro_f1': f1_score(y_true.flatten(), preds.flatten(), zero_division=0),
            'per_bm_f1': per_f1,
            'mean_auroc': np.nanmean([roc_auc_score(y_true[:,i], probs[:,i]) if len(np.unique(y_true[:,i]))>1 else np.nan for i in range(y_true.shape[1])])}

def optimize_thresholds(y_val, logits):
    probs = sigmoid(logits)
    best = np.zeros(y_val.shape[1])
    for i in range(y_val.shape[1]):
        best_f1, best_t = -1, 0.5
        for t in np.arange(0.05, 1.0, 0.05):
            f1 = f1_score(y_val[:,i], (probs[:,i]>=t).astype(int), zero_division=0)
            if f1 > best_f1: best_f1, best_t = f1, t
        best[i] = best_t
    return best

def compute_pos_weight(df_tr, bm_cols, device):
    pos = df_tr[bm_cols].sum().values; neg = len(df_tr) - pos
    return torch.tensor(np.clip(neg / np.clip(pos, 1, None), 0.05, 50), dtype=torch.float32, device=device)

@torch.no_grad()
def eval_model(model, loader, criterion, device):
    model.eval()
    total, n = 0.0, 0
    logits_all, labels_all = [], []
    for imgs, labs in loader:
        imgs, labs = imgs.to(device), labs.to(device)
        with torch.amp.autocast('cuda', enabled=device.type=='cuda'):
            out, _ = model(imgs); loss = criterion(out, labs)
        total += loss.item() * imgs.size(0); n += imgs.size(0)
        logits_all.append(out.float().cpu().numpy()); labels_all.append(labs.cpu().numpy())
    return total/max(n,1), np.concatenate(logits_all), np.concatenate(labels_all)

def train_model(name, gb, df_tr, df_va, df_te, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tr_ds = OCTDataset(df_tr, PATH_COL, biomarker_cols, train_transform)
    va_ds = OCTDataset(df_va, PATH_COL, biomarker_cols, eval_transform)
    te_ds = OCTDataset(df_te, PATH_COL, biomarker_cols, eval_transform)
    tr_dl = DataLoader(tr_ds, BATCH_SIZE, shuffle=True, num_workers=2)
    va_dl = DataLoader(va_ds, BATCH_SIZE, num_workers=2)
    te_dl = DataLoader(te_ds, BATCH_SIZE, num_workers=2)
    
    model = build_model(name, gb).to(DEVICE)
    pw = compute_pos_weight(df_tr, biomarker_cols, DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', 0.5, 2)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type=='cuda')
    best_vl, best_st, no_imp = float('inf'), copy.deepcopy(model.state_dict()), 0
    
    for ep in range(MAX_EPOCHS):
        model.train(); rl, ns = 0.0, 0
        for imgs, labs in tr_dl:
            imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=DEVICE.type=='cuda'):
                out, _ = model(imgs); loss = crit(out, labs)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update()
            rl += loss.item()*imgs.size(0); ns += imgs.size(0)
        vl, _, _ = eval_model(model, va_dl, nn.BCEWithLogitsLoss(), DEVICE)
        sched.step(vl)
        if vl < best_vl: best_vl = vl; best_st = copy.deepcopy(model.state_dict()); no_imp = 0
        else:
            no_imp += 1
            if no_imp >= EARLY_STOP_PATIENCE: break
    
    model.load_state_dict(best_st)
    _, vl_logits, vl_labels = eval_model(model, va_dl, nn.BCEWithLogitsLoss(), DEVICE)
    _, te_logits, te_labels = eval_model(model, te_dl, nn.BCEWithLogitsLoss(), DEVICE)
    thresholds = optimize_thresholds(vl_labels, vl_logits)
    metrics = compute_metrics(te_labels, te_logits, thresholds)
    return model, metrics, thresholds, te_logits, te_labels

print('Training functions defined.')

In [ ]:
# Run Layer 1 experiment
MODEL_NAMES = ['cnn', 'gcn', 'signed_gcn', 'prior_gat']
all_results = []
best_models = {}

for mname in MODEL_NAMES:
    for seed in SEEDS:
        print(f'\n{"="*60}\nTraining {mname} seed={seed}\n{"="*60}')
        model, metrics, thresholds, te_logits, te_labels = train_model(
            mname, graph_bundle, df_train, df_val, df_test, seed
        )
        print(f'  macro-F1={metrics["macro_f1"]:.4f}  AUROC={metrics["mean_auroc"]:.4f}')
        all_results.append({'model': mname, 'seed': seed, **metrics, 'thresholds': thresholds.tolist()})
        best_models[(mname, seed)] = model
        torch.save(model.state_dict(), os.path.join(CKPT_DIR, f'{mname}_seed{seed}.pt'))

results_df = pd.DataFrame(all_results)
print('\n' + '='*60)
print('Layer 1 Results (mean +/- SD across seeds):')
print('='*60)
for mname in MODEL_NAMES:
    sub = results_df[results_df['model']==mname]
    print(f'  {mname:15s}: macro-F1={sub["macro_f1"].mean():.4f}+/-{sub["macro_f1"].std():.4f}  AUROC={sub["mean_auroc"].mean():.4f}')

### 1.6 Layer 1 Results Visualization

In [ ]:
# Ablation bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels_map = {'cnn': 'CNN\n(independent)', 'gcn': 'GCN\n(+positive)', 'signed_gcn': 'Signed GCN\n(+negative)', 'prior_gat': 'Prior-GAT\n(+attention)'}

for ax, metric in zip(axes, ['macro_f1', 'mean_auroc']):
    means = [results_df[results_df['model']==m][metric].mean() for m in MODEL_NAMES]
    stds = [results_df[results_df['model']==m][metric].std() for m in MODEL_NAMES]
    bars = ax.bar([labels_map[m] for m in MODEL_NAMES], means, yerr=stds, capsize=5,
                  color=['#95a5a6', '#3498db', '#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.set_ylim(0, 1)
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.3f}', ha='center', fontsize=9)

fig.suptitle('Layer 1: Biomarker Detection — Model Ablation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(METRICS_DIR, 'layer1_ablation.png'), dpi=150)
plt.show()

---
# LAYER 2: Temporal Disease Progression

Graph-constrained LSTM that predicts how a patient's biomarker state evolves over time, using the co-occurrence structure from Layer 1.

In [ ]:
# Build longitudinal sequences from OLIVES visits
def build_sequences(df, patient_col, eye_col, visit_col, bm_cols):
    seqs = []
    group_cols = [patient_col] + ([eye_col] if eye_col else [])
    for key, gdf in df.groupby(group_cols):
        if not isinstance(key, tuple): key = (key,)
        if visit_col and visit_col in gdf.columns:
            gdf = gdf.sort_values(visit_col)
        if len(gdf) < 2: continue
        bm = gdf[bm_cols].values.astype(np.float32)
        # Clinical features: use BCVA and CST if available
        clin = np.zeros((len(gdf), 2), dtype=np.float32)
        for i, col_name in enumerate(['BCVA', 'CST']):
            matched = find_column(gdf, [col_name])
            if matched: clin[:, i] = gdf[matched].values.astype(np.float32)
        seqs.append({'bm': bm, 'clin': np.nan_to_num(clin), 'treat': np.zeros(len(gdf), dtype=np.int64)})
    return seqs

train_seqs = build_sequences(df_train, PATIENT_COL, EYE_COL, VISIT_COL, biomarker_cols)
val_seqs = build_sequences(df_val, PATIENT_COL, EYE_COL, VISIT_COL, biomarker_cols)
print(f'Longitudinal sequences: train={len(train_seqs)}, val={len(val_seqs)}')

# If not enough sequences, generate synthetic data
USE_SYNTHETIC = len(train_seqs) < 10
if USE_SYNTHETIC:
    print('Not enough longitudinal data — generating synthetic progression data for demo...')

In [ ]:
# Graph-Constrained Progression LSTM
class GraphProgressionLSTM(nn.Module):
    def __init__(self, n_bm=N_BM, n_clin=2, n_treat=5, hidden=PROG_HIDDEN, n_layers=PROG_LAYERS, npmi_mat=None):
        super().__init__()
        self.n_bm = n_bm
        self.treat_emb = nn.Embedding(n_treat, 16)
        inp_dim = n_bm + n_clin + 16
        self.lstm = nn.LSTM(inp_dim, hidden, n_layers, batch_first=True, dropout=0.2 if n_layers>1 else 0)
        self.node_proj = nn.Linear(hidden, hidden)
        if npmi_mat is not None:
            self.register_buffer('npmi', torch.as_tensor(npmi_mat, dtype=torch.float32))
        else:
            self.register_buffer('npmi', torch.zeros(n_bm, n_bm))
        self.gq = nn.Linear(hidden, hidden); self.gk = nn.Linear(hidden, hidden)
        self.gv = nn.Linear(hidden, hidden); self.gs = nn.Parameter(torch.tensor(0.5))
        self.gnorm = nn.LayerNorm(hidden)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden, 1))

    def forward(self, bm_seq, clin_seq, treat_seq):
        B, T, _ = bm_seq.shape
        x = torch.cat([bm_seq, clin_seq, self.treat_emb(treat_seq)], dim=-1)
        h, _ = self.lstm(x)
        h_flat = h.reshape(B*T, -1)
        nodes = self.node_proj(h_flat).unsqueeze(1).expand(-1, self.n_bm, -1)
        Q, K, V = self.gq(nodes), self.gk(nodes), self.gv(nodes)
        D = Q.shape[-1]
        scores = torch.einsum('bid,bjd->bij', Q, K)/math.sqrt(D) + self.gs * self.npmi.unsqueeze(0)
        attn = torch.softmax(scores, dim=-1)
        refined = F.relu(self.gnorm(nodes + torch.einsum('bij,bjd->bid', attn, V)))
        return self.head(refined).squeeze(-1).reshape(B, T, self.n_bm)

print('Graph-Constrained Progression LSTM defined.')

In [ ]:
# Progression dataset & training
class ProgDataset(Dataset):
    def __init__(self, seqs):
        self.data = []
        for s in seqs:
            bm, clin, treat = s['bm'], s['clin'], s['treat']
            if len(bm) < 2: continue
            self.data.append({
                'bm_in': torch.tensor(bm[:-1]), 'clin_in': torch.tensor(clin[:-1]),
                'treat_in': torch.tensor(treat[:-1]), 'bm_tgt': torch.tensor(bm[1:]), 'length': len(bm)-1,
            })
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

def collate_prog(batch):
    ml = max(b['length'] for b in batch); B = len(batch); nb = batch[0]['bm_in'].shape[-1]
    bm_in = torch.zeros(B, ml, nb); clin_in = torch.zeros(B, ml, 2)
    treat_in = torch.zeros(B, ml, dtype=torch.long); bm_tgt = torch.zeros(B, ml, nb)
    mask = torch.zeros(B, ml, dtype=torch.bool)
    for i, b in enumerate(batch):
        L = b['length']
        bm_in[i,:L] = b['bm_in']; clin_in[i,:L] = b['clin_in']
        treat_in[i,:L] = b['treat_in']; bm_tgt[i,:L] = b['bm_tgt']; mask[i,:L] = True
    return bm_in, clin_in, treat_in, bm_tgt, mask

def generate_synthetic_seqs(npmi_mat, n=200):
    rng = np.random.RandomState(42)
    base_p = np.array([0.4, 0.3, 0.5, 0.6, 0.7, 0.2])[:N_BM]
    seqs = []
    for i in range(n):
        T = rng.randint(3, 9); treat = rng.randint(0, 4)
        bm = np.zeros((T, N_BM), dtype=np.float32)
        bm[0] = (rng.random(N_BM) < base_p).astype(np.float32)
        for t in range(1, T):
            rate = 0.05 if treat < 2 else 0.1
            effect = -0.15 if treat == 0 else (-0.08 if treat == 1 else 0.0)
            probs = np.clip(bm[t-1] + rate + effect + rng.normal(0, 0.05, N_BM), 0, 1)
            if npmi_mat is not None:
                for j in range(N_BM):
                    for k in range(N_BM):
                        if j != k and abs(npmi_mat[j,k]) > 0.3:
                            adj = 0.1 if npmi_mat[j,k] > 0 else -0.1
                            if bm[t-1,k] > 0.5: probs[j] += adj
            bm[t] = (rng.random(N_BM) < np.clip(probs, 0, 1)).astype(np.float32)
        clin = rng.randn(T, 2).astype(np.float32) * np.array([10, 50]) + np.array([60, 300])
        seqs.append({'bm': bm, 'clin': clin, 'treat': np.full(T, treat, dtype=np.int64)})
    return seqs[:int(n*0.8)], seqs[int(n*0.8):]

# Use real or synthetic data
if USE_SYNTHETIC:
    synth_train, synth_val = generate_synthetic_seqs(npmi)
    prog_train_ds = ProgDataset(synth_train)
    prog_val_ds = ProgDataset(synth_val)
else:
    prog_train_ds = ProgDataset(train_seqs)
    prog_val_ds = ProgDataset(val_seqs)

print(f'Progression dataset: train={len(prog_train_ds)}, val={len(prog_val_ds)}')

In [ ]:
# Train progression model
prog_model = GraphProgressionLSTM(npmi_mat=npmi).to(DEVICE)
prog_opt = torch.optim.AdamW(prog_model.parameters(), lr=PROG_LR, weight_decay=1e-4)
prog_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(prog_opt, 'min', 0.5, 5)
prog_crit = nn.BCEWithLogitsLoss()

prog_tr_dl = DataLoader(prog_train_ds, 32, shuffle=True, collate_fn=collate_prog)
prog_va_dl = DataLoader(prog_val_ds, 32, collate_fn=collate_prog)

best_vl, best_st, no_imp = float('inf'), None, 0
for ep in range(PROG_EPOCHS):
    prog_model.train(); tl, ns = 0.0, 0
    for bm_in, cl_in, tr_in, bm_tgt, mask in prog_tr_dl:
        bm_in, cl_in, tr_in = bm_in.to(DEVICE), cl_in.to(DEVICE), tr_in.to(DEVICE)
        bm_tgt, mask = bm_tgt.to(DEVICE), mask.to(DEVICE)
        prog_opt.zero_grad()
        pred = prog_model(bm_in, cl_in, tr_in)
        loss = prog_crit(pred[mask], bm_tgt[mask])
        loss.backward(); nn.utils.clip_grad_norm_(prog_model.parameters(), 2.0)
        prog_opt.step()
        tl += loss.item()*mask.sum().item(); ns += mask.sum().item()
    tl /= max(ns, 1)
    
    prog_model.eval(); vl, vn = 0.0, 0
    with torch.no_grad():
        for bm_in, cl_in, tr_in, bm_tgt, mask in prog_va_dl:
            bm_in, cl_in, tr_in = bm_in.to(DEVICE), cl_in.to(DEVICE), tr_in.to(DEVICE)
            bm_tgt, mask = bm_tgt.to(DEVICE), mask.to(DEVICE)
            pred = prog_model(bm_in, cl_in, tr_in)
            loss = prog_crit(pred[mask], bm_tgt[mask])
            vl += loss.item()*mask.sum().item(); vn += mask.sum().item()
    vl /= max(vn, 1)
    prog_sched.step(vl)
    
    if vl < best_vl: best_vl = vl; best_st = copy.deepcopy(prog_model.state_dict()); no_imp = 0
    else:
        no_imp += 1
        if no_imp >= EARLY_STOP_PATIENCE: break
    if (ep+1) % 10 == 0:
        print(f'  Epoch {ep+1}: train_loss={tl:.4f} val_loss={vl:.4f}')

prog_model.load_state_dict(best_st)
torch.save(prog_model.state_dict(), os.path.join(CKPT_DIR, 'progression_model.pt'))
print(f'Progression model trained. Best val_loss={best_vl:.4f}')

---
# LAYER 3: Virtual Drug Testing

Treatment simulation engine that uses the progression model to compare counterfactual treatment scenarios.

In [ ]:
class TreatmentSimulator:
    def __init__(self, model, device):
        self.model = model; self.device = device; self.model.eval()
    
    @torch.no_grad()
    def simulate(self, initial_bm, initial_clin, treatment_idx, n_steps=6):
        traj = [initial_bm.copy()]; probs_list = [initial_bm.copy()]
        cur_bm = torch.tensor(initial_bm, dtype=torch.float32).reshape(1,1,-1).to(self.device)
        cur_clin = torch.tensor(initial_clin, dtype=torch.float32).reshape(1,1,-1).to(self.device)
        for step in range(n_steps):
            treat = torch.tensor([[treatment_idx]], dtype=torch.long, device=self.device)
            logits = self.model(cur_bm, cur_clin, treat)
            probs = torch.sigmoid(logits[:,-1,:]).cpu().numpy()[0]
            binary = (probs > 0.5).astype(np.float32)
            traj.append(binary); probs_list.append(probs)
            cur_bm = torch.tensor(binary, dtype=torch.float32).reshape(1,1,-1).to(self.device)
            effect = np.array([0.5, -5.0]) if treatment_idx > 0 else np.array([-0.3, 3.0])
            new_clin = initial_clin + effect*(step+1)
            cur_clin = torch.tensor(new_clin, dtype=torch.float32).reshape(1,1,-1).to(self.device)
        return np.array(traj), np.array(probs_list), np.mean(np.array(probs_list), axis=1)
    
    def compare_all(self, initial_bm, initial_clin, n_steps=6):
        results = {}
        for i, t in enumerate(TREATMENTS):
            traj, probs, severity = self.simulate(initial_bm, initial_clin, i+1, n_steps)
            results[t] = {'trajectory': traj, 'probabilities': probs, 'severity': severity}
        # Add observation baseline
        traj, probs, severity = self.simulate(initial_bm, initial_clin, 0, n_steps)
        results['observation'] = {'trajectory': traj, 'probabilities': probs, 'severity': severity}
        # Find best
        final_sev = {t: r['severity'][-1] for t, r in results.items()}
        results['_best'] = min(final_sev, key=final_sev.get)
        results['_final_severity'] = final_sev
        return results

simulator = TreatmentSimulator(prog_model, DEVICE)
print('Treatment simulator ready.')

In [ ]:
# Demo: simulate treatments for a synthetic patient
demo_bm = np.array([1, 0, 1, 1, 1, 0], dtype=np.float32)  # IRHRF, FAVF, IRF, DRT/ME active
demo_clin = np.array([55.0, 340.0], dtype=np.float32)

results = simulator.compare_all(demo_bm, demo_clin, n_steps=6)

print(f'Recommended treatment: {results["_best"]}')
print(f'\nFinal severity by treatment:')
for t, s in sorted(results['_final_severity'].items(), key=lambda x: x[1]):
    marker = ' << BEST' if t == results['_best'] else ''
    print(f'  {t:15s}: {s:.4f}{marker}')

In [ ]:
# Visualize treatment comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Severity trajectories
colors = {'anti_vegf': '#2ecc71', 'steroid': '#3498db', 'laser': '#e67e22', 'observation': '#95a5a6'}
for t in TREATMENTS + ['observation']:
    r = results[t]
    style = '--' if t == 'observation' else '-'
    lw = 1.5 if t == 'observation' else 2.5
    axes[0].plot(r['severity'], style, color=colors.get(t, '#333'), linewidth=lw, label=t, marker='o', markersize=4)
axes[0].set_xlabel('Visit'); axes[0].set_ylabel('Severity Score')
axes[0].set_title('Disease Severity Under Different Treatments')
axes[0].legend(loc='best'); axes[0].set_ylim(0, 1); axes[0].grid(True, alpha=0.3)

# Final severity bar chart
treats = TREATMENTS + ['observation']
final_sevs = [results[t]['severity'][-1] for t in treats]
bar_colors = ['#2ecc71' if t == results['_best'] else '#95a5a6' for t in treats]
axes[1].bar(treats, final_sevs, color=bar_colors, edgecolor='white', linewidth=1.5)
axes[1].set_ylabel('Final Severity'); axes[1].set_title('Final Severity Comparison')
for i, (t, s) in enumerate(zip(treats, final_sevs)):
    axes[1].text(i, s+0.02, f'{s:.3f}', ha='center', fontsize=9)
axes[1].set_ylim(0, 1)

plt.suptitle('Layer 3: Virtual Drug Testing Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(METRICS_DIR, 'layer3_treatment_comparison.png'), dpi=150)
plt.show()

---
# DIGITAL TWIN: End-to-End Pipeline

Putting it all together: OCT → biomarkers → progression → treatment recommendation

In [ ]:
# Select best Layer 1 model
best_row = max(all_results, key=lambda r: r['macro_f1'])
best_l1 = best_models[(best_row['model'], best_row['seed'])]
print(f'Best Layer 1: {best_row["model"]} seed={best_row["seed"]} F1={best_row["macro_f1"]:.4f}')

# Digital Twin demo on test patients
test_patients = df_test[PATIENT_COL].unique()[:5]
print(f'\nRunning Digital Twin on {len(test_patients)} test patients:\n')

for pid in test_patients:
    patient_df = df_test[df_test[PATIENT_COL] == pid]
    bm_vals = patient_df[biomarker_cols].iloc[0].values.astype(np.float32)
    active = [BIOMARKERS[i] for i in range(N_BM) if bm_vals[i] > 0]
    
    # Simulate treatments
    clin = np.array([60.0, 300.0], dtype=np.float32)
    comparison = simulator.compare_all(bm_vals, clin, n_steps=6)
    
    print(f'Patient {pid}:')
    print(f'  Active biomarkers: {active}')
    print(f'  Severity: {np.mean(bm_vals):.3f}')
    print(f'  Recommended: {comparison["_best"]}')
    print(f'  Final severity: {comparison["_final_severity"][comparison["_best"]]:.3f}')
    print()

In [ ]:
# Per-biomarker trajectory visualization for one patient
pid = test_patients[0]
bm_vals = df_test[df_test[PATIENT_COL]==pid][biomarker_cols].iloc[0].values.astype(np.float32)
clin = np.array([55.0, 340.0])

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (bm_name, ax) in enumerate(zip(BIOMARKERS, axes.flat)):
    for t in TREATMENTS:
        r = results[t]
        ax.plot(r['probabilities'][:, i], marker='o', markersize=3, label=t, linewidth=1.5)
    ax.set_title(bm_name, fontweight='bold')
    ax.set_xlabel('Visit'); ax.set_ylabel('P(active)')
    ax.set_ylim(-0.05, 1.05); ax.grid(True, alpha=0.3)
    if i == 0: ax.legend(fontsize=7)

plt.suptitle(f'Per-Biomarker Trajectory — Patient {pid}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(METRICS_DIR, 'per_biomarker_trajectory.png'), dpi=150)
plt.show()

---
# Summary & Paper-Ready Report

In [ ]:
report = {
    'project': 'Digital Twin for Diabetic Retinopathy',
    'subtitle': 'Graph-Aware Biomarker Detection, Progression Modeling & Virtual Drug Testing',
    'dataset': {'name': 'OLIVES (PRIME + TREX DME)', 'total_images': len(df), 'patients': df[PATIENT_COL].nunique()},
    'layer1_biomarker_detection': {
        'models': MODEL_NAMES,
        'best_model': best_row['model'],
        'best_macro_f1': best_row['macro_f1'],
        'best_auroc': best_row['mean_auroc'],
        'results_by_model': {m: {'macro_f1': results_df[results_df['model']==m]['macro_f1'].mean(),
                                  'std': results_df[results_df['model']==m]['macro_f1'].std()}
                             for m in MODEL_NAMES},
    },
    'layer2_progression': {
        'model': 'Graph-Constrained LSTM',
        'uses_npmi_graph': True,
        'best_val_loss': best_vl,
    },
    'layer3_treatment': {
        'treatments_compared': TREATMENTS + ['observation'],
        'demo_recommendation': results['_best'],
        'demo_final_severity': results['_final_severity'],
    },
    'co_occurrence_graph': {
        'method': 'NPMI (train-only)',
        'top_k': GRAPH_TOP_K,
        'strongest_positive': 'IRF ↔ DRT/ME',
        'strongest_negative': 'PAVF ↔ FAVF',
    },
}

with open(os.path.join(METRICS_DIR, 'final_report.json'), 'w') as f:
    json.dump(report, f, indent=2, default=str)

print(json.dumps(report, indent=2, default=str))
print(f'\nReport saved to {METRICS_DIR}/final_report.json')

In [ ]:
print('\n' + '='*70)
print('DIGITAL TWIN PIPELINE COMPLETE')
print('='*70)
print(f'\nLayer 1: Best biomarker model = {best_row["model"]} (macro-F1 = {best_row["macro_f1"]:.4f})')
print(f'Layer 2: Progression model val_loss = {best_vl:.4f}')
print(f'Layer 3: Recommended treatment = {results["_best"]}')
print(f'\nOutputs saved to: {OUT_DIR}/')
print(f'  checkpoints/  — trained model weights')
print(f'  metrics/      — results, plots, and final report')
print(f'\nTo launch the interactive UI:')
print(f'  streamlit run ui/app.py')